# Evaluacion Final - OHR Sobre CICIDS

Su objetivo no es solo entrenar modelos, sino responder de forma ordenada a las preguntas centrales del trabajo:

1. que ocurre al comparar un **flujo estructurado** como OHR frente a modelos monoliticos sobre datos tabulares;
2. si OHR mantiene rendimiento predictivo competitivo;
3. si OHR aporta ventajas de eficiencia en inferencia;
4. que informacion adicional aportan sus metricas internas sobre el proceso de decision.

La idea es que un tercero pueda ejecutar este notebook con:

- una carpeta local que contenga los CSV de CICIDS2017;
- la libreria `ohr` instalada;
- dependencias comunes de analisis y machine learning.


## 1. Dependencias Y Parametros

Dependencias necesarias:

- `ohr`
- `pandas`
- `numpy`
- `scikit-learn`
- `matplotlib`
- `seaborn`

La celda siguiente define la ruta del dataset, la escala de ejecucion y algunos parametros del experimento.


In [ ]:
from __future__ import annotations

import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from ohr import OHRClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 40)

# Cambia esta ruta para apuntar a la carpeta que contiene todos los CSV de CICIDS2017.
DATASET_DIR = Path(r"C:\Users\jose_\Documents\PROYECTOS\PERSONAL\MAESTRIA_IA_INTERNACIONES\TESIS\desarrollo\external_data\CICIDS2017")

# "reduced" sirve para validacion rapida. "full" usa todo el dataset.
RUN_SCALE = "reduced"

# "global_stratified" es el protocolo principal. "source_aware_stratified"
# conserva mejor la senal de procedencia por archivo.
SPLIT_PROTOCOL = "global_stratified"

SEED = 42
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
REDUCED_CAP_PER_LARGE_CLASS = 20_000
SOURCE_AWARE_MIN_GROUP_SIZE = 5

# Parametros de exportacion.
RUN_NAME = f"cicids_tesis_{RUN_SCALE}_{SPLIT_PROTOCOL}"
EXPORT_RESULTS = True
OUTPUT_DIR = Path.cwd() / "tesis_notebook_outputs" / RUN_NAME

# Parametros de entrenamiento de OHR.
OHR_BATCH_SIZE = 1024
OHR_EPOCHS = 30 if RUN_SCALE == "full" else 20
PRIMARY_OHR_CONFIG = "ohr_cicids_real_sharp_routing"
BASELINE_OHR_REFERENCE = "ohr_cicids_real_base"

print(f"DATASET_DIR: {DATASET_DIR}")
print(f"RUN_SCALE: {RUN_SCALE}")
print(f"SPLIT_PROTOCOL: {SPLIT_PROTOCOL}")
print(f"RUN_NAME: {RUN_NAME}")
print(f"PRIMARY_OHR_CONFIG: {PRIMARY_OHR_CONFIG}")


## 2. Utilidades Minimas

Para mantener el notebook simple, solo se definen unas pocas funciones auxiliares:

- normalizacion ligera de etiquetas;
- construccion segura de la clave de estratificacion;
- conversion de scores a probabilidades aproximadas para `LinearSVC`;
- evaluacion uniforme de modelos;
- exportacion de tablas Markdown sin depender de `tabulate`.


In [ ]:
def normalize_label(value: object) -> str:
    text = str(value).strip()
    return " ".join(text.split())


def build_stratify_key(labels: pd.Series, sources: pd.Series, protocol: str) -> pd.Series | None:
    labels = labels.astype(str)
    if protocol == "global_stratified":
        return labels
    combo = labels + "||" + sources.astype(str)
    combo_counts = combo.value_counts()
    fallback = combo.map(combo_counts).lt(SOURCE_AWARE_MIN_GROUP_SIZE)
    key = combo.where(~fallback, labels)
    if key.value_counts().min() < 2:
        return labels if labels.value_counts().min() >= 2 else None
    return key


def decision_scores_to_probabilities(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


def evaluate_predictions(
    *,
    model_name: str,
    config_name: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    probabilities: np.ndarray | None,
    train_seconds: float,
    inference_seconds: float,
    extra_metrics: dict | None = None,
) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    extra_metrics = extra_metrics or {}
    confidence = (
        probabilities.max(axis=1)
        if probabilities is not None
        else np.full(shape=len(y_pred), fill_value=np.nan, dtype=np.float64)
    )
    report = pd.DataFrame(classification_report(y_true, y_pred, output_dict=True, zero_division=0)).T
    matrix = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=sorted(pd.unique(y_true))),
        index=sorted(pd.unique(y_true)),
        columns=sorted(pd.unique(y_true)),
    )
    row = {
        "model": model_name,
        "config_name": config_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "train_seconds": train_seconds,
        "inference_seconds": inference_seconds,
        "mean_confidence": float(np.nanmean(confidence)),
    }
    row.update(extra_metrics)
    audit = pd.DataFrame(
        {
            "true_label": y_true,
            "predicted_label": y_pred,
            "confidence": confidence,
            "correct": y_true == y_pred,
            "model": model_name,
            "config_name": config_name,
        }
    )
    return row, report, matrix, audit


def frame_to_markdown(frame: pd.DataFrame) -> str:
    def format_value(value: object) -> str:
        if isinstance(value, (list, tuple, np.ndarray)):
            return json.dumps(np.asarray(value).tolist(), ensure_ascii=False)
        if isinstance(value, dict):
            return json.dumps(value, ensure_ascii=False)
        if value is None:
            return ""
        if isinstance(value, (float, np.floating)):
            if math.isnan(float(value)):
                return ""
            return f"{float(value):.6f}"
        return str(value)

    columns = [str(column) for column in frame.columns]
    rows = [[format_value(value) for value in row] for row in frame.to_numpy(dtype=object)]
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header, separator, *body])


## 3. Descubrimiento Del Dataset

Antes de entrenar, conviene verificar que:

- la carpeta contiene realmente CSVs;
- el esquema sea consistente;
- la columna objetivo exista;
- el problema se pueda tratar como multiclase real.


In [ ]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"No existe la carpeta del dataset: {DATASET_DIR}")

csv_files = sorted(DATASET_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No se encontraron archivos CSV en: {DATASET_DIR}")

dataset_inventory = pd.DataFrame(
    {
        "file_name": [path.name for path in csv_files],
        "size_mb": [round(path.stat().st_size / (1024 * 1024), 2) for path in csv_files],
    }
)
display(dataset_inventory)
print(f"CSV encontrados: {len(csv_files)}")


In [ ]:
schema_rows = []
for path in csv_files:
    sample = pd.read_csv(path, nrows=1000, low_memory=False)
    sample.columns = [column.strip() for column in sample.columns]
    schema_rows.append(
        {
            "file_name": path.name,
            "n_columns": sample.shape[1],
            "has_label": "Label" in sample.columns,
            "sample_dtypes": dict(sample.dtypes.astype(str).value_counts()),
        }
    )

schema_overview = pd.DataFrame(schema_rows)
display(schema_overview)


## 4. Carga Y Limpieza Estructural

En esta fase se realiza solo limpieza **segura y no aprendida**:

- unificacion de columnas;
- eliminacion de una columna duplicada conocida;
- eliminacion de columnas constantes conocidas;
- reemplazo de infinitos por `NaN`;
- normalizacion de etiquetas;
- conversion de features numericas a `float32` para ahorrar memoria.


In [ ]:
raw_frames = []
for path in csv_files:
    frame = pd.read_csv(path, low_memory=False)
    frame.columns = [column.strip() for column in frame.columns]
    frame["source_file"] = path.name
    raw_frames.append(frame)

data = pd.concat(raw_frames, ignore_index=True)
print(f"Filas totales cargadas: {len(data):,}")
print(f"Columnas originales: {data.shape[1]}")


In [ ]:
target_column = "Label"
source_column = "source_file"

if target_column not in data.columns:
    raise KeyError(f"No se encontro la columna objetivo '{target_column}'")

duplicate_columns_to_drop = ["Fwd Header Length.1"]
constant_zero_columns = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
]
columns_to_drop = [column for column in duplicate_columns_to_drop + constant_zero_columns if column in data.columns]
if columns_to_drop:
    data = data.drop(columns=columns_to_drop)

data[target_column] = data[target_column].map(normalize_label)

feature_columns = [column for column in data.columns if column not in {target_column, source_column}]
data[feature_columns] = data[feature_columns].replace([np.inf, -np.inf], np.nan)
data[feature_columns] = data[feature_columns].apply(pd.to_numeric, errors="coerce").astype(np.float32)

missing_summary = data[feature_columns].isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary.gt(0)].head(10)

print(f"Columnas finales tras limpieza estructural: {data.shape[1]}")
print(f"Numero de clases: {data[target_column].nunique()}")
display(missing_summary.to_frame("missing_values"))


In [ ]:
class_distribution = (
    data[target_column]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)
display(class_distribution.head(20))


## 5. Reduccion Opcional Del Dataset

El modo `reduced` sirve para validar el experimento con menor costo computacional.

La idea es:

- conservar todas las clases;
- mantener todas las observaciones de clases pequenas;
- limitar solo las clases mas grandes para acelerar la corrida.


In [ ]:
working_data = data.copy()

if RUN_SCALE == "reduced":
    sampled_parts = []
    rng = np.random.default_rng(SEED)
    for label, group in working_data.groupby(target_column, sort=False):
        if len(group) <= REDUCED_CAP_PER_LARGE_CLASS:
            sampled_parts.append(group)
            continue
        sampled_index = rng.choice(group.index.to_numpy(), size=REDUCED_CAP_PER_LARGE_CLASS, replace=False)
        sampled_parts.append(group.loc[np.sort(sampled_index)])
    working_data = pd.concat(sampled_parts, ignore_index=True)

working_distribution = (
    working_data[target_column]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

print(f"Escala activa: {RUN_SCALE}")
print(f"Filas usadas en esta corrida: {len(working_data):,}")
display(working_distribution.head(20))


## 6. Split Reproducible Sin Data Leakage

Todos los modelos van a reutilizar exactamente el mismo split.

Se separa primero `test`, y luego `validation`, para obtener un esquema:

- train: 70%
- validation: 15%
- test: 15%


In [ ]:
labels = working_data[target_column].astype(str)
sources = working_data[source_column].astype(str)
stratify_key = build_stratify_key(labels, sources, SPLIT_PROTOCOL)

X_full = working_data.drop(columns=[target_column])
y_full = labels.copy()

test_fraction = TEST_SIZE
remaining_fraction = 1.0 - test_fraction
validation_fraction_within_remaining = VAL_SIZE / remaining_fraction

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_full,
    y_full,
    test_size=test_fraction,
    random_state=SEED,
    stratify=stratify_key,
)

stratify_key_trainval = build_stratify_key(
    y_trainval.reset_index(drop=True),
    X_trainval[source_column].reset_index(drop=True),
    SPLIT_PROTOCOL,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=validation_fraction_within_remaining,
    random_state=SEED,
    stratify=stratify_key_trainval,
)

print("Tamano de splits")
print(f"Train: {len(X_train):,}")
print(f"Validation: {len(X_val):,}")
print(f"Test: {len(X_test):,}")


## 7. Preparacion Final De Features

A partir de aqui:

- OHR recibe `DataFrame` tabular con su columna tecnica `source_file` ya retirada;
- los baselines reciben las mismas features, pero con pipelines especificos si estan justificados.


In [ ]:
feature_columns = [column for column in X_train.columns if column != source_column]

X_train_model = X_train[feature_columns].reset_index(drop=True)
X_val_model = X_val[feature_columns].reset_index(drop=True)
X_test_model = X_test[feature_columns].reset_index(drop=True)

y_train_model = y_train.reset_index(drop=True)
y_val_model = y_val.reset_index(drop=True)
y_test_model = y_test.reset_index(drop=True)

print(f"Numero de features usadas: {len(feature_columns)}")
print(f"NaN en train: {int(X_train_model.isna().sum().sum()):,}")
print(f"NaN en val: {int(X_val_model.isna().sum().sum()):,}")
print(f"NaN en test: {int(X_test_model.isna().sum().sum()):,}")


## 8. Configuraciones De OHR

Se usan dos variantes:

- **OHR sharp routing**: variante principal del notebook y configuracion OHR recomendada;
- **OHR base**: referencia estable e interpretable para comparacion interna.

Esto permite responder no solo la comparacion con modelos monoliticos, sino tambien una comparacion interna de OHR, tomando `sharp_routing` como candidato principal.


In [ ]:
ohr_base_config = {
    "tabularizer": {"enabled": True, "input_type": "tabular", "replace_infinite": True},
    "preprocessing": {"handle_missing": "median", "scaling": "standard"},
    "embedding": {
        "enabled": True,
        "mode": "pca_based",
        "explained_variance_ratio": 0.95,
        "whiten": False,
        "random_state": SEED,
    },
    "adapter": {"type": "linear", "hidden_dims": [], "dropout": 0.0},
    "projection": {"type": "learnable", "apply_to": "fused"},
    "routing": {
        "mode": "soft",
        "depth": 3,
        "temperature": 1.2,
        "temperature_schedule": "linear",
        "temperature_end": 0.9,
        "temperature_schedule_epochs": 10,
    },
    "expert": {"type": "linear", "hidden_dims": [], "dropout": 0.0, "use_projected_features": True},
    "aggregator": {"type": "weighted_logits"},
    "training": {
        "batch_size": OHR_BATCH_SIZE,
        "epochs": OHR_EPOCHS,
        "lr": 0.001,
        "weight_decay": 0.0001,
        "early_stopping_patience": 6,
        "orthogonal_regularization_weight": 0.001,
        "diversity_regularization_weight": 0.001,
        "load_balance_weight": 0.001,
        "regularization_schedule": "linear_warmup",
        "regularization_warmup_epochs": 6,
        "class_weighting": "balanced_sqrt",
        "class_weight_cap": 2.0,
        "classification_loss": "cross_entropy",
        "focal_gamma": 2.0,
        "label_smoothing": 0.02,
        "confidence_penalty_weight": 0.0,
        "inference_temperature": 1.05,
        "gradient_clip_norm": 1.0,
        "device": "cpu",
        "end_to_end": True,
        "selection_metric": "f1_macro",
    },
    "seed": SEED,
}

ohr_sharp_routing_config = json.loads(json.dumps(ohr_base_config))
ohr_sharp_routing_config["routing"]["temperature_end"] = 0.7

ohr_variants = [
    ("ohr_cicids_real_sharp_routing", ohr_sharp_routing_config),
    ("ohr_cicids_real_base", ohr_base_config),
]

display(pd.DataFrame(
    [
        {"config_name": name, "temperature_end": config["routing"]["temperature_end"], "depth": config["routing"]["depth"]}
        for name, config in ohr_variants
    ]
))


## 9. Modelos Comparados

Los modelos comparados en este notebook son:

- `OHR base`
- `OHR sharp routing`
- `RandomForest`
- `LogisticRegression`
- `LinearSVC`

`OHR sharp routing` sigue siendo la variante **principal** del experimento, pero se mantiene `OHR base` dentro del listado para que la comparacion interna de OHR quede visible desde el principio.


In [ ]:
compared_models_catalog = pd.DataFrame(
    [
        {"family": "OHR", "config_name": "ohr_cicids_real_base", "role": "referencia interna"},
        {"family": "OHR", "config_name": "ohr_cicids_real_sharp_routing", "role": "variante OHR principal"},
        {"family": "Baseline", "config_name": "RandomForest", "role": "modelo monolitico de referencia"},
        {"family": "Baseline", "config_name": "LogisticRegression", "role": "modelo lineal de referencia"},
        {"family": "Baseline", "config_name": "LinearSVC", "role": "modelo lineal max-margin"},
    ]
)
display(compared_models_catalog)


## 10. Baselines Monoliticos

Los modelos de referencia son:

- `RandomForest`
- `LogisticRegression`
- `LinearSVC`

Los pipelines se mantienen simples y justificables:

- imputacion mediana para todos;
- escalado solo donde es razonable (`LogisticRegression` y `LinearSVC`);
- `RandomForest` sin escalado.


In [ ]:
baseline_models = {
    "RandomForest": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=150 if RUN_SCALE == "reduced" else 200,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "LogisticRegression": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    solver="saga",
                    max_iter=500,
                    class_weight="balanced",
                    random_state=SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "LinearSVC": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LinearSVC(
                    C=1.0,
                    max_iter=6000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    ),    
}
list(baseline_models.keys())


## 11. Entrenamiento Y Evaluacion De Baselines

En esta seccion se entrena cada baseline sobre el mismo `train`, y se evalua en el mismo `test`.


In [ ]:
results_rows = []
confusion_matrices: dict[str, pd.DataFrame] = {}
class_reports: dict[str, pd.DataFrame] = {}
prediction_audits: list[pd.DataFrame] = []

for model_name, pipeline in baseline_models.items():
    start_train = time.perf_counter()
    pipeline.fit(X_train_model, y_train_model)
    train_seconds = time.perf_counter() - start_train

    start_inference = time.perf_counter()
    y_pred = pipeline.predict(X_test_model)
    inference_seconds = time.perf_counter() - start_inference

    probabilities = None
    if hasattr(pipeline, "predict_proba"):
        probabilities = pipeline.predict_proba(X_test_model)
    else:
        scores = pipeline.decision_function(X_test_model)
        probabilities = decision_scores_to_probabilities(scores)

    row, report, matrix, audit = evaluate_predictions(
        model_name=model_name,
        config_name=model_name,
        y_true=y_test_model.to_numpy(),
        y_pred=np.asarray(y_pred),
        probabilities=probabilities,
        train_seconds=train_seconds,
        inference_seconds=inference_seconds,
    )
    results_rows.append(row)
    class_reports[model_name] = report
    confusion_matrices[model_name] = matrix
    prediction_audits.append(audit)

pd.DataFrame(results_rows)


## 12. Entrenamiento Y Evaluacion De OHR

Aqui se entrena OHR con sus dos variantes.

Ademas de las metricas externas habituales, se guardan:

- metricas internas del routing;
- metadata de la corrida;
- historial de entrenamiento.


In [ ]:
ohr_histories: dict[str, pd.DataFrame] = {}
ohr_metadata: dict[str, dict] = {}
ohr_routing_metrics: dict[str, dict] = {}

for config_name, config in ohr_variants:
    model = OHRClassifier(config)

    start_train = time.perf_counter()
    history = model.fit(
        X_train_model,
        y_train_model,
        validation_data=(X_val_model, y_val_model),
        epochs=OHR_EPOCHS,
        batch_size=OHR_BATCH_SIZE,
        random_state=SEED,
    )
    train_seconds = time.perf_counter() - start_train

    start_inference = time.perf_counter()
    probabilities = model.predict_proba(X_test_model)
    y_pred = model.predict_labels(X_test_model)
    inference_seconds = time.perf_counter() - start_inference

    routing_diagnostics = model.get_routing_diagnostics(X_test_model, top_k=3)
    routing_metrics = routing_diagnostics["routing_metrics"]

    row, report, matrix, audit = evaluate_predictions(
        model_name="OHR",
        config_name=config_name,
        y_true=y_test_model.to_numpy(),
        y_pred=np.asarray(y_pred),
        probabilities=probabilities,
        train_seconds=train_seconds,
        inference_seconds=inference_seconds,
        extra_metrics={
            "routing_entropy": routing_metrics.get("routing_entropy"),
            "mean_leaf_probability": routing_metrics.get("mean_leaf_probability"),
            "load_balance_score": routing_metrics.get("load_balance_score"),
            "load_balance_mse": routing_metrics.get("load_balance_mse"),
            "effective_experts": routing_metrics.get("effective_experts"),
            "mean_effective_depth": routing_metrics.get("mean_effective_depth"),
            "mean_top_expert_probability": routing_metrics.get("mean_top_expert_probability"),
            "mean_projection_penalty": routing_metrics.get("mean_projection_penalty"),
        },
    )

    results_rows.append(row)
    class_reports[config_name] = report
    confusion_matrices[config_name] = matrix
    prediction_audits.append(audit)
    ohr_histories[config_name] = history.to_frame()
    ohr_metadata[config_name] = model.get_run_metadata()
    ohr_routing_metrics[config_name] = routing_metrics

results_frame = pd.DataFrame(results_rows).sort_values(
    by=["f1_macro", "accuracy"],
    ascending=False,
).reset_index(drop=True)
display(results_frame)


## 13. Comparacion Global De Resultados

Esta tabla responde directamente a la comparacion principal de la tesis:

- rendimiento predictivo;
- eficiencia computacional;
- posicion relativa de OHR frente a modelos monoliticos.


In [ ]:
display(
    results_frame[
        [
            "model",
            "config_name",
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "f1_weighted",
            "train_seconds",
            "inference_seconds",
            "mean_confidence",
        ]
    ]
)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.barplot(data=results_frame, x="config_name", y="accuracy", ax=axes[0, 0], palette="deep")
axes[0, 0].set_title("Accuracy")
axes[0, 0].tick_params(axis="x", rotation=45)

sns.barplot(data=results_frame, x="config_name", y="f1_macro", ax=axes[0, 1], palette="deep")
axes[0, 1].set_title("F1 Macro")
axes[0, 1].tick_params(axis="x", rotation=45)

sns.barplot(data=results_frame, x="config_name", y="train_seconds", ax=axes[1, 0], palette="deep")
axes[1, 0].set_title("Tiempo de entrenamiento (s)")
axes[1, 0].tick_params(axis="x", rotation=45)

sns.barplot(data=results_frame, x="config_name", y="inference_seconds", ax=axes[1, 1], palette="deep")
axes[1, 1].set_title("Tiempo de inferencia (s)")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 14. Comparacion De Predicciones

La tesis no se beneficia solo de una tabla agregada. Tambien conviene revisar:

- como se distribuyen los aciertos y errores;
- que modelo domina por cantidad de predicciones correctas;
- donde OHR mejora o empeora frente a los baselines.


In [ ]:
prediction_audit = pd.concat(prediction_audits, ignore_index=True)

verification_summary = (
    prediction_audit.groupby(["model", "config_name"], as_index=False)
    .agg(
        total_predictions=("correct", "size"),
        correct_predictions=("correct", "sum"),
        audit_accuracy=("correct", "mean"),
        mean_confidence=("confidence", "mean"),
        confidence_when_correct=("confidence", lambda values: float(np.nanmean(values[prediction_audit.loc[values.index, "correct"]]))),
        confidence_when_wrong=("confidence", lambda values: float(np.nanmean(values[~prediction_audit.loc[values.index, "correct"]]))),
    )
    .sort_values(by="audit_accuracy", ascending=False)
)
display(verification_summary)


In [ ]:
rf_audit = prediction_audit.query("config_name == 'RandomForest'").reset_index(drop=True)
ohr_audit = prediction_audit.query("config_name == @PRIMARY_OHR_CONFIG").reset_index(drop=True)

pairwise_prediction_review = pd.DataFrame(
    {
        "true_label": rf_audit["true_label"],
        "random_forest_prediction": rf_audit["predicted_label"],
        "random_forest_correct": rf_audit["correct"],
        "ohr_prediction": ohr_audit["predicted_label"],
        "ohr_correct": ohr_audit["correct"],
    }
)
pairwise_prediction_review["comparison_case"] = np.select(
    [
        pairwise_prediction_review["random_forest_correct"] & pairwise_prediction_review["ohr_correct"],
        (~pairwise_prediction_review["random_forest_correct"]) & pairwise_prediction_review["ohr_correct"],
        pairwise_prediction_review["random_forest_correct"] & (~pairwise_prediction_review["ohr_correct"]),
    ],
    [
        "ambos_correctos",
        "solo_ohr_correcto",
        "solo_random_forest_correcto",
    ],
    default="ambos_incorrectos",
)

display(pairwise_prediction_review["comparison_case"].value_counts().to_frame("count"))


## 15. Matrices De Confusion Y Reportes Por Clase

Esta seccion ayuda a responder si los modelos se comportan igual en todas las clases o si algunas categorias son especialmente dificiles.


In [ ]:
selected_models = ["RandomForest", PRIMARY_OHR_CONFIG, BASELINE_OHR_REFERENCE]

fig, axes = plt.subplots(1, len(selected_models), figsize=(7 * len(selected_models), 6))
if len(selected_models) == 1:
    axes = [axes]

for axis, name in zip(axes, selected_models):
    matrix = confusion_matrices[name]
    sns.heatmap(matrix, cmap="Blues", ax=axis, cbar=False)
    axis.set_title(name)
    axis.set_xlabel("Prediccion")
    axis.set_ylabel("Real")

plt.tight_layout()
plt.show()


In [ ]:
rf_report = class_reports["RandomForest"].reset_index(names="label")
ohr_base_report = class_reports[BASELINE_OHR_REFERENCE].reset_index(names="label")
ohr_sharp_report = class_reports[PRIMARY_OHR_CONFIG].reset_index(names="label")

comparison_by_class = (
    ohr_sharp_report[["label", "precision", "recall", "f1-score", "support"]]
    .rename(
        columns={
            "precision": "ohr_precision",
            "recall": "ohr_recall",
            "f1-score": "ohr_f1",
            "support": "support",
        }
    )
    .merge(
        rf_report[["label", "precision", "recall", "f1-score"]].rename(
            columns={
                "precision": "rf_precision",
                "recall": "rf_recall",
                "f1-score": "rf_f1",
            }
        ),
        on="label",
        how="left",
    )
)
comparison_by_class["ohr_minus_rf_f1"] = comparison_by_class["ohr_f1"] - comparison_by_class["rf_f1"]
comparison_by_class = comparison_by_class.sort_values(by="ohr_minus_rf_f1", ascending=False)
display(comparison_by_class.head(15))
display(comparison_by_class.tail(15))


## 16. Metricas Internas De OHR

Esta es una de las diferencias clave entre OHR y los modelos monoliticos.

Ademas de las metricas externas, OHR permite observar:

- entropia del routing;
- balance de expertos;
- profundidad efectiva;
- intensidad de la proyeccion.


In [ ]:
ohr_internal_frame = (
    results_frame.query("model == 'OHR'")
    [
        [
            "config_name",
            "accuracy",
            "f1_macro",
            "routing_entropy",
            "mean_leaf_probability",
            "load_balance_score",
            "load_balance_mse",
            "effective_experts",
            "mean_effective_depth",
            "mean_top_expert_probability",
            "mean_projection_penalty",
        ]
    ]
    .reset_index(drop=True)
)
display(ohr_internal_frame)


## 17. Curvas De Entrenamiento De OHR

Las curvas permiten observar si la variante recomendada mejora por una razon coherente, por ejemplo:

- menor perdida;
- mejor `val_f1_macro`;
- routing mas decidido sin colapso.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for config_name, history in ohr_histories.items():
    axes[0, 0].plot(history["epoch"], history["train_loss"], label=config_name)
    axes[0, 1].plot(history["epoch"], history["val_accuracy"], label=config_name)
    axes[1, 0].plot(history["epoch"], history["val_f1_macro"], label=config_name)
    axes[1, 1].plot(history["epoch"], history["val_routing_entropy"], label=config_name)

axes[0, 0].set_title("Train loss")
axes[0, 1].set_title("Validation accuracy")
axes[1, 0].set_title("Validation F1 macro")
axes[1, 1].set_title("Validation routing entropy")

for axis in axes.flat:
    axis.set_xlabel("Epoch")
    axis.legend()

plt.tight_layout()
plt.show()


## 18. Validaciones Finales




In [ ]:
validation_rows = []

for _, row in results_frame.iterrows():
    config_name = row["config_name"]
    matrix = confusion_matrices[config_name]
    computed_accuracy = np.trace(matrix.to_numpy()) / matrix.to_numpy().sum()
    validation_rows.append(
        {
            "config_name": config_name,
            "reported_accuracy": row["accuracy"],
            "accuracy_from_confusion_matrix": computed_accuracy,
            "accuracy_delta": abs(row["accuracy"] - computed_accuracy),
            "test_samples": int(matrix.to_numpy().sum()),
        }
    )

validation_frame = pd.DataFrame(validation_rows)
display(validation_frame)


In [ ]:
best_accuracy_row = results_frame.sort_values(by="accuracy", ascending=False).iloc[0]
best_f1_row = results_frame.sort_values(by="f1_macro", ascending=False).iloc[0]
fastest_training_row = results_frame.sort_values(by="train_seconds", ascending=True).iloc[0]
fastest_inference_row = results_frame.sort_values(by="inference_seconds", ascending=True).iloc[0]
best_ohr_row = results_frame.query("model == 'OHR'").sort_values(by="f1_macro", ascending=False).iloc[0]

conclusion = {
    "best_accuracy": {
        "model": best_accuracy_row["model"],
        "config_name": best_accuracy_row["config_name"],
        "accuracy": float(best_accuracy_row["accuracy"]),
    },
    "best_f1_macro": {
        "model": best_f1_row["model"],
        "config_name": best_f1_row["config_name"],
        "f1_macro": float(best_f1_row["f1_macro"]),
    },
    "fastest_training": {
        "model": fastest_training_row["model"],
        "config_name": fastest_training_row["config_name"],
        "train_seconds": float(fastest_training_row["train_seconds"]),
    },
    "fastest_inference": {
        "model": fastest_inference_row["model"],
        "config_name": fastest_inference_row["config_name"],
        "inference_seconds": float(fastest_inference_row["inference_seconds"]),
    },
    "best_ohr_variant": {
        "config_name": best_ohr_row["config_name"],
        "accuracy": float(best_ohr_row["accuracy"]),
        "f1_macro": float(best_ohr_row["f1_macro"]),
    },
}

display(Markdown(
    f"""
### Lectura preliminar

- El mejor modelo en **accuracy** fue **{best_accuracy_row["config_name"]}**.
- El mejor modelo en **F1 macro** fue **{best_f1_row["config_name"]}**.
- El entrenamiento mas rapido fue **{fastest_training_row["config_name"]}**.
- La inferencia mas rapida fue **{fastest_inference_row["config_name"]}**.
- La mejor variante de OHR en esta corrida fue **{best_ohr_row["config_name"]}**.
"""
))


## 19. Exportacion Opcional

Si `EXPORT_RESULTS = True`, el notebook deja una carpeta con tablas y artefactos utiles para anexos, revision externa o redaccion de resultados.


In [ ]:
if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results_frame.to_csv(OUTPUT_DIR / "master_results.csv", index=False)
    (OUTPUT_DIR / "master_results.md").write_text(frame_to_markdown(results_frame), encoding="utf-8")

    verification_summary.to_csv(OUTPUT_DIR / "prediction_verification_summary.csv", index=False)
    prediction_audit.to_csv(OUTPUT_DIR / "prediction_audit.csv", index=False)
    comparison_by_class.to_csv(OUTPUT_DIR / "comparison_by_class.csv", index=False)
    validation_frame.to_csv(OUTPUT_DIR / "validation_checks.csv", index=False)

    for name, frame in confusion_matrices.items():
        safe_name = name.lower().replace(" ", "_")
        frame.to_csv(OUTPUT_DIR / f"confusion_matrix_{safe_name}.csv")

    for name, frame in class_reports.items():
        safe_name = name.lower().replace(" ", "_")
        frame.to_csv(OUTPUT_DIR / f"classification_report_{safe_name}.csv", index=True)

    for name, history in ohr_histories.items():
        safe_name = name.lower().replace(" ", "_")
        history.to_csv(OUTPUT_DIR / f"history_{safe_name}.csv", index=False)

    (OUTPUT_DIR / "summary.json").write_text(
        json.dumps(conclusion, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Resultados exportados a: {OUTPUT_DIR.resolve()}")
